In [2]:
# ============================================================
# STEP 1: Upload File & Load Dataset
# ============================================================
import pandas as pd
import numpy as np
from google.colab import files

# This will open a file picker — select your iris.csv
uploaded = files.upload()

# Load the dataset (automatically picks the uploaded filename)
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("=== First 5 Rows ===")
print(df.head())

print("\n=== Shape (rows, columns) ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Basic Statistics ===")
print(df.describe())

print("\n=== Missing Values Per Column ===")
print(df.isnull().sum())

print("\n=== Class Distribution (species) ===")
print(df['species'].value_counts())

Saving 1) iris.csv to 1) iris (1).csv
=== First 5 Rows ===
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

=== Shape (rows, columns) ===
(150, 5)

=== Data Types ===
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species          object
dtype: object

=== Basic Statistics ===
       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.054000      3.758667     1.198667
std        0.828066     0.433594      1.764420     0.763161
min        4.300000     2.000000      1.000000     0.100000
25%        5.100000     2.800000     

In [3]:
# ============================================================
# STEP 2: Handle Missing Data
# ============================================================

# --- Simulate some missing values for demonstration ---
df_missing = df.copy()
np.random.seed(42)

# Randomly inject ~10% missing values into numerical columns
for col in ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']:
    missing_idx = np.random.choice(df_missing.index, size=15, replace=False)
    df_missing.loc[missing_idx, col] = np.nan

print("Missing values after injection:")
print(df_missing.isnull().sum())

# --- Strategy 1: Fill numerical columns with MEAN ---
df_filled_mean = df_missing.copy()
for col in ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']:
    mean_val = df_filled_mean[col].mean()
    df_filled_mean[col].fillna(mean_val, inplace=True)

print("\nAfter filling with MEAN — missing values:")
print(df_filled_mean.isnull().sum())

# --- Strategy 2: Fill numerical columns with MEDIAN ---
df_filled_median = df_missing.copy()
for col in ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']:
    median_val = df_filled_median[col].median()
    df_filled_median[col].fillna(median_val, inplace=True)

print("\nAfter filling with MEDIAN — missing values:")
print(df_filled_median.isnull().sum())

# --- Strategy 3: Drop rows with any missing values ---
df_dropped = df_missing.dropna()
print(f"\nRows after dropping NaN rows: {len(df_dropped)} (original: {len(df_missing)})")

# We'll use the mean-filled version going forward
df_clean = df_filled_mean.copy()
print("\nUsing mean-filled dataset. Final missing values:", df_clean.isnull().sum().sum())

Missing values after injection:
sepal_length    15
sepal_width     15
petal_length    15
petal_width     15
species          0
dtype: int64

After filling with MEAN — missing values:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

After filling with MEDIAN — missing values:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

Rows after dropping NaN rows: 98 (original: 150)

Using mean-filled dataset. Final missing values: 0


/tmp/ipykernel_24296/19062475.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filled_mean[col].fillna(mean_val, inplace=True)
/tmp/ipykernel_24296/19062475.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [4]:
# ============================================================
# STEP 3: Encode Categorical Variables
# ============================================================

# --- Method 1: Label Encoding (Setosa=0, Versicolor=1, Virginica=2) ---
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_label = df_clean.copy()
df_label['species_label'] = le.fit_transform(df_label['species'])

print("=== Label Encoding ===")
print(df_label[['species', 'species_label']].drop_duplicates())
print("\nLabel classes:", list(le.classes_))

# --- Method 2: One-Hot Encoding ---
df_onehot = df_clean.copy()
df_onehot = pd.get_dummies(df_onehot, columns=['species'], prefix='species')

print("\n=== One-Hot Encoding — New Columns ===")
print(df_onehot[['species_setosa', 'species_versicolor', 'species_virginica']].head(5))
print("\nShape after one-hot encoding:", df_onehot.shape)

# We'll use Label Encoding for the final pipeline (common for classification targets)
df_encoded = df_label.copy()
df_encoded.drop(columns=['species'], inplace=True)  # Drop original text column
print("\nFinal encoded dataframe columns:", df_encoded.columns.tolist())
print(df_encoded.head())

=== Label Encoding ===
        species  species_label
0        setosa              0
50   versicolor              1
100   virginica              2

Label classes: ['setosa', 'versicolor', 'virginica']

=== One-Hot Encoding — New Columns ===
   species_setosa  species_versicolor  species_virginica
0            True               False              False
1            True               False              False
2            True               False              False
3            True               False              False
4            True               False              False

Shape after one-hot encoding: (150, 7)

Final encoded dataframe columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species_label']
   sepal_length  sepal_width  petal_length  petal_width  species_label
0           5.1          3.5           1.4          0.2              0
1           4.9          3.0           1.4          0.2              0
2           4.7          3.2           1.3        

In [5]:
# ============================================================
# STEP 4: Normalize / Standardize Numerical Features
# ============================================================
from sklearn.preprocessing import MinMaxScaler, StandardScaler

feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# --- Method 1: Normalization (Min-Max Scaling → range [0, 1]) ---
min_max_scaler = MinMaxScaler()
df_normalized = df_encoded.copy()
df_normalized[feature_cols] = min_max_scaler.fit_transform(df_encoded[feature_cols])

print("=== After Min-Max Normalization ===")
print(df_normalized[feature_cols].describe().round(4))

# --- Method 2: Standardization (Z-score → mean=0, std=1) ---
std_scaler = StandardScaler()
df_standardized = df_encoded.copy()
df_standardized[feature_cols] = std_scaler.fit_transform(df_encoded[feature_cols])

print("\n=== After Standardization (Z-score) ===")
print(df_standardized[feature_cols].describe().round(4))

# We'll use Standardization for the final split (generally preferred for ML)
df_final = df_standardized.copy()
print("\nFinal dataset preview:")
print(df_final.head())

=== After Min-Max Normalization ===
       sepal_length  sepal_width  petal_length  petal_width
count      150.0000     150.0000      150.0000     150.0000
mean         0.4259       0.4324        0.4603       0.4361
std          0.2196       0.1687        0.2857       0.2999
min          0.0000       0.0000        0.0000       0.0000
25%          0.2292       0.3333        0.1017       0.0833
50%          0.4259       0.4167        0.5254       0.4583
75%          0.5764       0.5000        0.6907       0.6563
max          1.0000       1.0000        1.0000       1.0000

=== After Standardization (Z-score) ===
       sepal_length  sepal_width  petal_length  petal_width
count      150.0000     150.0000      150.0000     150.0000
mean         0.0000      -0.0000        0.0000      -0.0000
std          1.0034       1.0034        1.0034       1.0034
min         -1.9463      -2.5722       -1.6163      -1.4590
25%         -0.8991      -0.5893       -1.2591      -1.1802
50%          0.0000    

In [6]:
# ============================================================
# STEP 5: Split into Training and Testing Sets
# ============================================================
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_final[feature_cols]
y = df_final['species_label']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:\n", y.value_counts())

# Split: 80% train, 20% test (stratified to keep class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y         # ensures equal class proportion in both sets
)

print("\n=== Split Results ===")
print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Testing set:    X={X_test.shape},  y={y_test.shape}")

print("\nTraining class distribution:\n", y_train.value_counts())
print("\nTesting class distribution:\n",  y_test.value_counts())

Features shape: (150, 4)
Target shape: (150,)
Target distribution:
 species_label
0    50
1    50
2    50
Name: count, dtype: int64

=== Split Results ===
Training set:   X=(120, 4), y=(120,)
Testing set:    X=(30, 4),  y=(30,)

Training class distribution:
 species_label
0    40
2    40
1    40
Name: count, dtype: int64

Testing class distribution:
 species_label
0    10
2    10
1    10
Name: count, dtype: int64


In [7]:
# ============================================================
# STEP 6: Final Summary
# ============================================================

print("=" * 50)
print("       PREPROCESSING COMPLETE — SUMMARY")
print("=" * 50)
print(f"Original dataset shape     : {df.shape}")
print(f"After handling missing data: {df_clean.shape}")
print(f"Encoding method used       : Label Encoding")
print(f"Scaling method used        : StandardScaler (Z-score)")
print(f"Training samples           : {X_train.shape[0]}")
print(f"Testing samples            : {X_test.shape[0]}")
print(f"Number of features         : {X_train.shape[1]}")
print(f"Number of classes          : {y.nunique()} {list(le.classes_)}")
print("=" * 50)
print("\nX_train sample:")
print(X_train.head(3))
print("\ny_train sample:")
print(y_train.head(3).values)
print("\nData is ready for Machine Learning! ✅")

       PREPROCESSING COMPLETE — SUMMARY
Original dataset shape     : (150, 5)
After handling missing data: (150, 5)
Encoding method used       : Label Encoding
Scaling method used        : StandardScaler (Z-score)
Training samples           : 120
Testing samples            : 30
Number of features         : 4
Number of classes          : 3 ['setosa', 'versicolor', 'virginica']

X_train sample:
     sepal_length  sepal_width  petal_length  petal_width
8       -1.819357    -0.341488     -1.378181    -1.319609
106     -1.184698    -1.332906      0.466889     0.000000
76       0.000000    -0.589343      0.645444     0.353135

y_train sample:
[0 2 1]

Data is ready for Machine Learning! ✅
